In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Day2Practice") \
    .getOrCreate()

employees = spark.createDataFrame([
    (1, "Ravi", "IT", 75000, "2021-05-10"),
    (2, "Rahul", "IT", 60000, "2022-03-15"),
    (3, "Priya", "HR", 65000, "2020-08-20"),
    (4, "Anu", "HR", 55000, "2023-01-10"),
    (5, "Kiran", "Finance", 90000, "2019-11-05"),
    (6, "Sneha", "Finance", 70000, "2022-07-18"),
    (7, "Arjun", "IT", None, "2024-02-12"),
    (8, "Meena", "HR", 65000, None),
    (9, "Vikram", "Finance", 85000, "2021-09-25"),
    (10, "Riya", "Sales", 50000, "2023-06-30")
], ["id", "name", "department", "salary", "joining_date"])

employees.show()
employees.printSchema()

In [0]:
customers = spark.createDataFrame([
    (1, "Ravi"),
    (2, "Rahul"),
    (3, "Priya"),
    (4, "Anu"),
    (5, "Kiran")
], ["customer_id", "name"])

customers.show()

In [0]:
orders = spark.createDataFrame([
    (101, 1, 500),
    (102, 1, 700),
    (103, 2, 300),
    (104, 3, 1000),
    (105, 3, 500),
    (106, 6, 900)
], ["order_id", "customer_id", "amount"])

orders.show()

In [0]:
employees.select("name","department","salary").show()

In [0]:
employees.withColumnsRenamed({
    "name": "employee_name",
    "department": "employee_department"
}).show()

In [0]:
employees.select(
    "id",
    col("name").alias("employee_name"),
    col("salary").alias("annual_salary")
).show()

In [0]:
employees.withColumn("monthly_salary",col("salary")/12).show()

In [0]:
employees.withColumn("salary_with_column",(col("salary")+(col("salary")*0.1))).show()

In [0]:
employees=employees.withColumn("salary",col("salary").cast("double"))

In [0]:
employees.printSchema()

In [0]:
employees.filter(col("salary")>60000).show()

In [0]:
employees.filter(col("department").isin("IT","HR")).show()

In [0]:
employees.filter(col("salary").between(60000,80000)).show()

In [0]:
employees.filter(col("salary").isNull()).show()

In [0]:
employees.orderBy(col("salary").desc()).show()

In [0]:
employees.orderBy(
    col("department").asc(),
    col("salary").desc()
).show()

In [0]:
employees.dropna(subset="salary").show()

In [0]:
employees.fillna(50000,subset="salary").show()

In [0]:
employees.withColumn("clean_name",trim(lower(col("name")))).show()

In [0]:
employees.printSchema()

In [0]:
employees=employees.withColumn("joining_date",to_date(col("joining_date")))

In [0]:
employees.printSchema()

In [0]:
employees.withColumn("joining_year",year(col("joining_date"))).show()

In [0]:
employees.withColumn("joining_month",month(col("joining_date"))).show()

### Aggregations

In [0]:
employees.agg(
    avg("salary").alias("average_salary")
).show()

In [0]:
employees.groupBy("department").agg(
    avg("salary").alias("average_salary_dept")
).show()

In [0]:
employees.groupBy("department").agg(
    max("salary").alias("max_salary_dept")
).show()

In [0]:
employees.groupBy("department").agg(
    min("salary").alias("min_salary_dept")
).show()

In [0]:
employees.groupBy("department").agg(
    count("id").alias("count_dept")
).show()

In [0]:
employees.groupBy("department").agg(
    sum(col("salary")).alias("total_sal_dept")
).show()

In [0]:
employees.groupBy("department").agg(
    count("id").alias("employee_count"),
    avg("salary").alias("average_salary"),
    max("salary").alias("max_salary"),
    min("salary").alias("min_salary"),
    sum(col("salary")).alias("total_salary")
).show()

In [0]:
employees.groupBy("department")\
.agg(avg("salary").alias("average_salary"))\
    .filter(col("average_salary")>80000)\
    .show()

In [0]:
window_func=Window.orderBy(col("average_salary").desc())
employees.groupBy("department")\
.agg(
    avg("salary").alias("average_salary")
    )\
    .withColumn("rank",dense_rank().over(window_func))\
    .filter(col("rank")==1)\
    .drop("rank")\
    .show()

### Union

In [0]:
data2 = [
    (11, "Amit", "IT", 72000.0, "2023-08-15"),
    (12, "Neha", "HR", 58000.0, "2022-11-20"),
    (13, "Rajesh", "Finance", 88000.0, "2021-04-05"),
    (14, "Pooja", "Marketing", 62000.0, "2024-01-10"),
    (1, "Ravi", "IT", 75000.0, "2021-05-10")
]

schema = ["id", "name", "department", "salary", "joining_date"]

employees2 = spark.createDataFrame(data2, schema)
employees2 = employees2.withColumn("joining_date", to_date(col("joining_date")))

In [0]:
employees.union(employees2).show()

### Joins

In [0]:
res=customers.join(
    orders,
    customers.customer_id==orders.customer_id,
    "inner"
)

res.show()

### Window functions

In [0]:
window_func=Window.partitionBy("department").orderBy("salary")

employees.withColumn("rank",dense_rank().over(window_func)).show()
